In [3]:
# !uv add requests beautifulsoup4 scikit-learn

In [3]:
import requests
from bs4 import BeautifulSoup
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# !uv add msal
import msal

In [5]:
CLIENT_ID = "ab564a04-c563-4d89-be61-05c7883653dc"

AUTHORITY = "https://login.microsoftonline.com/consumers"

SCOPES = ["Notes.Read"]

In [6]:
app = msal.PublicClientApplication(
    client_id=CLIENT_ID,
    authority=AUTHORITY
)

In [7]:
flow = app.initiate_device_flow(
    scopes=SCOPES
)
# print(flow)
print(flow['message'])


To sign in, use a web browser to open the page https://www.microsoft.com/link and enter the code DV5MAHCE to authenticate.


In [8]:
result = app.acquire_token_by_device_flow(flow)

In [13]:
if "access_token" in result:
    access_token = result["access_token"]
    print("Authentication successfuls!")
else:
    print("Authentication failed:")
    print(result.get("error"))
    print(result.get("error_description"))

Authentication successfuls!


In [ ]:
# import requests

# headers = {
#     "Authorization": f"Bearer {access_token}"
# }

# response = requests.get(
#     "https://graph.microsoft.com/v1.0/me/onenote/notebooks",
#     headers=headers
# )

# response.raise_for_status()

# notebooks = response.json()


In [22]:
# print(notebooks["value"])

In [ ]:
# for notebook in notebooks["value"]:
#     print("Notebook:", notebook["displayName"])
#     # print("ID:", notebook["id"])
#     # print("-" * 50)

Notebook: testnote


In [ ]:
# testnote = None

# for notebook in notebooks["value"]:
#     if notebook["displayName"] == "testnote":
#         #getting the notebook object:
#         testnote = notebook
#         break

# print(testnote["displayName"])

testnote


In [ ]:
# notebook_id = testnote["id"]

In [14]:
def read_onenote(access_token, notebook_name):

    headers = {
        "Authorization": f"Bearer {access_token}"
    }

    base_url = "https://graph.microsoft.com/v1.0/me/onenote"

    documents = []

    # Get all notebooks
    notebooks_response = requests.get(
        f"{base_url}/notebooks",
        headers=headers
    )

    notebooks_response.raise_for_status()
    notebooks = notebooks_response.json()["value"]

    # Find the notebook we want
    notebook = None

    for item in notebooks:
        if item["displayName"] == notebook_name:
            notebook = item
            break

    # Stop if the notebook was not found
    if notebook is None:
        raise ValueError(
            f"Notebook '{notebook_name}' was not found."
        )

    notebook_id = notebook["id"]

    # Get sections belonging only to this notebook
    sections_response = requests.get(
        f"{base_url}/notebooks/{notebook_id}/sections",
        headers=headers
    )

    sections_response.raise_for_status()
    sections = sections_response.json()["value"]

    # Go through each section
    for section in sections:

        section_id = section["id"]
        section_name = section["displayName"]

        # Get pages belonging to this section
        pages_response = requests.get(
            f"{base_url}/sections/{section_id}/pages",
            headers=headers
        )

        pages_response.raise_for_status()
        pages = pages_response.json()["value"]

        # Go through each page
        for page in pages:

            page_id = page["id"]
            page_title = page["title"]

            # Get the actual page content
            content_response = requests.get(
                f"{base_url}/pages/{page_id}/content",
                headers=headers
            )

            content_response.raise_for_status()
            page_html = content_response.text

            # Convert OneNote HTML into searchable text
            soup = BeautifulSoup(
                page_html,
                "html.parser"
            )

            page_text = soup.get_text(
                separator=" ",
                strip=True
            )

            # Convert the OneNote page into our document format
            document = {
                "id": page_id,
                "notebook": notebook_name,
                "section": section_name,
                "title": page_title,
                "content": page_text,
                "source_type": "onenote",
                "last_modified": page.get("lastModifiedDateTime")
            }

            documents.append(document)

    return documents

In [15]:
documents = read_onenote(
    access_token,
    notebook_name="testnote"
)
len(documents)

5

In [17]:
for document in documents:
    print(
        document["notebook"],
        "→",
        document["section"],
        "→",
        document["title"]
    )

testnote → Desktop Support → Protecting University Computers
testnote → Desktop Support → Computer Will Not Start
testnote → Desktop Support → Maintaining Your Windows Desktop
testnote → IT Policies → Password and Account Security
testnote → IT Policies → Acceptable Use Policy


In [52]:
documents[2]

{'id': '0-ebbf72f227f646c89334a7f9efbde27c!1-3EFF8F91DE152C4E!sda6fbd34bc1b42a7b0a6c34a928393ff',
 'notebook': 'testnote',
 'section': 'Desktop Support',
 'title': 'Maintaining Your Windows Desktop',
 'content': 'Maintaining Your Windows Desktop Regular Maintenance Restart the computer regularly so operating-system updates can complete. Employees should avoid leaving university computers running for long periods without restarting them. Windows Update should be allowed to install approved security and system updates. Storage Users should periodically remove unnecessary downloads and temporary files. Important university documents should be stored in approved university storage locations rather than only on the computer\'s local drive. Employees should maintain sufficient free disk space for Windows updates and normal application operation. Performance Problems If the computer becomes unusually slow: Restart the computer. Check whether Windows Update is currently installing updates. Clo

In [ ]:

class OneNoteSearchEngine:

    def __init__(self, documents):

        if not documents:
            raise ValueError("No documents were provided.")

        self.documents = documents

        # Build the searchable text for each document
        self.searchable_text = []

        for document in documents:

            text = (
                f"{document.get('notebook', '')} "
                f"{document.get('section', '')} "
                f"{document.get('title', '')} "
                f"{document.get('content', '')}"
            )

            self.searchable_text.append(text)

        # Create the TF-IDF vectorizer
        self.vectorizer = TfidfVectorizer(
            stop_words="english",
            lowercase=True
        )

        # Build the document index ONCE
        self.document_matrix = self.vectorizer.fit_transform(
            self.searchable_text
        )


    def search(self, query, num_results=5):

        # Validate the query
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query must contain text.")

        if num_results <= 0:
            raise ValueError("num_results must be greater than 0.")

        # Convert the query using the SAME vectorizer
        query_vector = self.vectorizer.transform([query])

        # Compare query against all indexed documents
        scores = cosine_similarity(
            self.document_matrix,
            query_vector
        ).flatten()

        # Sort highest score first
        best_indexes = scores.argsort()[::-1]

        results = []

        for index in best_indexes:

            # Ignore documents with zero similarity
            if scores[index] <= 0:
                continue

            result = self.documents[index].copy()

            result["score"] = float(scores[index])

            results.append(result)

            if len(results) >= num_results:
                break

        return results

In [45]:
#build the search index once
search_engine = OneNoteSearchEngine(documents)

In [50]:
for document in documents:
    for image in document.get("images", []):
        if "J. Rivera" in image.get("ocr_text", ""):
            print("FOUND!")
            print("Page:", document["title"])
            print("OCR:", image["ocr_text"])

In [47]:
#Now you can search
results = search_engine.search(
    "What should I do if my computer is running slowly?"
)


for position, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"RESULT {position}")
    print("=" * 80)

    print(f"Notebook : {result['notebook']}")
    print(f"Section  : {result['section']}")
    print(f"Title    : {result['title']}")
    print(f"Score    : {result['score']:.3f}")
    print(f"Modified : {result['last_modified']}")

    print()
    print("Content:")
    print(result["content"])

    print()

RESULT 1
Notebook : testnote
Section  : Desktop Support
Title    : Computer Will Not Start
Score    : 0.181
Modified : 2026-09-30T22:00:46.446Z

Content:
Computer Will Not Start Initial Troubleshooting If a desktop computer will not start, first verify that the power cable is securely connected to both the computer and the electrical outlet. Check whether the monitor and computer have power indicators. If a laptop will not start, connect its approved charger and wait several minutes before attempting to turn it on again. Computer Has Power but Windows Does Not Load If the computer powers on but Windows does not load normally, record any error message displayed on the screen. Do not repeatedly force the computer off and on. Contact IT support and provide: Computer asset number Location Description of the problem Error message, if available Approximate time the problem started

RESULT 2
Notebook : testnote
Section  : Desktop Support
Title    : Maintaining Your Windows Desktop
Score    : 

In [54]:
results = search_engine.search(
    "My computer has power but Windows will not load"
)


for position, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"RESULT {position}")
    print("=" * 80)

    print(f"Notebook : {result['notebook']}")
    print(f"Section  : {result['section']}")
    print(f"Title    : {result['title']}")
    print(f"Score    : {result['score']:.3f}")
    print(f"Modified : {result['last_modified']}")

    print()
    print("Content:")
    print(result["content"])

    print()

RESULT 1
Notebook : testnote
Section  : Desktop Support
Title    : Computer Will Not Start
Score    : 0.392
Modified : 2026-09-30T22:00:46.446Z

Content:
Computer Will Not Start Initial Troubleshooting If a desktop computer will not start, first verify that the power cable is securely connected to both the computer and the electrical outlet. Check whether the monitor and computer have power indicators. If a laptop will not start, connect its approved charger and wait several minutes before attempting to turn it on again. Computer Has Power but Windows Does Not Load If the computer powers on but Windows does not load normally, record any error message displayed on the screen. Do not repeatedly force the computer off and on. Contact IT support and provide: Computer asset number Location Description of the problem Error message, if available Approximate time the problem started

RESULT 2
Notebook : testnote
Section  : Desktop Support
Title    : Maintaining Your Windows Desktop
Score    : 